In [ ]:
from transformers import AutoModel,AutoTokenizer
import torch
import torch.nn.functional as F
model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)
documents=[
    "減脂晚餐建議吃高蛋白、低油脂、適量碳水",
    "胸肌訓練可以安排槓鈴握推、上斜啞鈴握推、夾胸",
    "深蹲可以訓練股四頭肌、臀部與核心穩定",
    "增肌期需要足夠蛋白質、熱量盈餘及重量練"
]
def encode_texts(texts):
    inputs=tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=model(**inputs)

    token_embeddings=outputs.last_hidden_state
    attention_mask=inputs["attention_mask"]

    mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings=token_embeddings*mask

    sum_embeddings=torch.sum(masked_embeddings,dim=1)
    sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

    sentence_embeddings=sum_embeddings/sum_mask
    sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)
    return sentence_embeddings

        
documents_embeddings=encode_texts(documents)
query="我現在減脂,晚餐吃甚麼比較好"
query_embeddings=encode_texts([query])
scores=torch.matmul(query_embeddings,documents_embeddings.T)
print("documents_embeddings shape:",documents_embeddings.shape)
print("query_embeddings shape:",query_embeddings.shape)
print("scores shape:",scores.shape)
print("scores:",scores)
topk_values,topk_indices=torch.topk(scores,k=2,dim=1)
print("\nTop Results:")
for i in range(topk_indices.shape[1]):
    idx=topk_indices[0][i].item()
    score=topk_values[0][i].item()
    print(f"Rank{i+1}:")
    print("documents",documents[idx])
    print("score",round(score,4))
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11833.53it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


documents_embeddings shape: torch.Size([4, 384])
query_embeddings shape: torch.Size([1, 384])
scores shape: torch.Size([1, 4])
scores: tensor([[0.7473, 0.0773, 0.0882, 0.3200]])

Top Results:
Rank1:
documents 減脂晚餐建議吃高蛋白、低油脂、適量碳水
score 0.7473

Rank2:
documents 增肌期需要足夠蛋白質、熱量盈餘及重量練
score 0.32



In [ ]:
"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
"減脂晚餐建議吃高蛋白、低油脂、適量碳水"
"胸肌訓練可以安排槓鈴握推、上斜啞鈴握推、夾胸"
"深蹲可以訓練股四頭肌、臀部與核心穩定"
"增肌期需要足夠蛋白質、熱量盈餘及重量練"
"我現在減脂,晚餐吃甚麼比較好"